TQC algorithm adapted from: https://github.com/SamsungLabs/tqc_pytorch/tree/master

D2RL code based this paper: https://arxiv.org/abs/2010.09163

D2RL code adapted from: https://github.com/pairlab/d2rl

ERE Buffer code based on this paper: https://arxiv.org/abs/1906.04009

**Dependencies and setup**

This can take a minute or so...

In [ ]:
%%capture
!pip install setuptools==65.5.0 "wheel<0.40.0"
!apt update
!apt-get install python3-opengl
!apt install xvfb -y
!pip install 'swig'
!pip install 'pyglet==1.5.27'
!pip install 'gym[box2d]==0.20.0'
!pip install 'pyvirtualdisplay==3.0'
!pip install torch
!pip install matplotlib

import gym
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import Module, Linear
from torch.nn.functional import logsigmoid
from torch.distributions import Normal, Distribution
import copy
import sys
from pyvirtualdisplay import Display
from IPython import display as disp
%matplotlib inline

display = Display(visible=0,size=(600,600))
display.start()
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

plot_interval = 10
video_every = 75

**Reinforcement learning agent**

In [ ]:
LOG_STD_MIN_MAX = (-20, 2)

class Critic(Module):
    def __init__(self, obs_dim, act_dim, n_quantiles, n_nets):
        super().__init__()
        self.nets = []
        self.n_quantiles = n_quantiles
        self.n_nets = n_nets

        for i in range(n_nets):
            net = MlpForCritic(obs_dim + act_dim, [256, 256], n_quantiles)
            self.add_module(f'qf{i}', net)
            self.nets.append(net)

    def forward(self, state, action):
        sa = torch.cat((state, action), dim=1)
        quantiles = torch.stack(tuple(net(sa) for net in self.nets), dim=1)
        return quantiles

class Actor(Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = MlpForActor(obs_dim, [512, 512], act_dim)

    def forward(self, obs):
        mean, log_std = self.net(obs)
        log_std = log_std.clamp(*LOG_STD_MIN_MAX)

        if self.training:
            std = torch.exp(log_std)
            tanh_normal = TanhNormal(mean, std)
            action, pre_tanh = tanh_normal.rsample()
            log_prob = tanh_normal.log_prob(pre_tanh)
            log_prob = log_prob.sum(dim=1, keepdim=True) 

        else: 
            action = torch.tanh(mean)
            log_prob = None
            
        return action, log_prob

    def select_action(self, obs):
        obs = torch.FloatTensor(obs).to(device)[np.newaxis, :]
        action, _ = self.forward(obs)
        action = action[0].cpu().detach().numpy()
        return action


class MlpForActor(Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()
        self.list_of_layers = []
        input_size_ = input_size
        num_inputs = 24 
        input_dim = hidden_sizes[0] + num_inputs
        for i, next_size in enumerate(hidden_sizes):
            if i == 0:
                lay = Linear(input_size_, next_size).to(device)
            else:
                  lay = Linear(input_dim, next_size).to(device)
            self.add_module(f'layer{i}', lay)
            self.list_of_layers.append(lay)
            input_size_ = next_size
            
        self.last_layer_mean_linear = Linear(input_dim, output_size).to(device)
        self.last_layer_log_std_linear = Linear(input_dim, output_size).to(device)

    def forward(self, input_):
        curr = input_

        for layer in self.list_of_layers:
            intermediate = layer(curr)
            curr = torch.nn.functional.gelu(intermediate)

            curr = torch.cat([curr, input_], dim=1)

        mean_linear = self.last_layer_mean_linear(curr)
        log_std_linear = self.last_layer_log_std_linear(curr)
        return mean_linear, log_std_linear


class MlpForCritic(Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()
        input_size_ = input_size
        input_dim = 28 + hidden_sizes[0] 
        self.list_of_layers = []
        for i, next_size in enumerate(hidden_sizes):
            if i == 0:
                  lay = Linear(input_size_, next_size).to(device)
            else: 
                  lay = Linear(input_dim, next_size).to(device)
            self.add_module(f'layer{i}', lay)
            self.list_of_layers.append(lay)
        self.last_layer = Linear(input_dim, output_size).to(device)
    

    def forward(self, input_):
        curr = input_
        for lay in self.list_of_layers:
            curr_ = torch.nn.functional.gelu(lay(curr))
            curr = torch.cat([curr_, input_], dim=1)
        output = self.last_layer(curr)
        return output
    
class TanhNormal(Distribution):
    def __init__(self, normal_mean, normal_std):
        super().__init__()
        self.normal_mean = normal_mean
        self.normal_std = normal_std
        self.standard_normal = Normal(torch.zeros_like(self.normal_mean, device=device),
                                      torch.ones_like(self.normal_std, device=device))
        self.normal = Normal(normal_mean, normal_std)
    
    def logsigmoid(tensor):
        denominator = 1 + torch.exp(-tensor)
        return torch.log(1/ denominator)

    def log_prob(self, pre_tanh):
        log_det = 2 * np.log(2) + logsigmoid(2 * pre_tanh) + logsigmoid(-2 * pre_tanh)
        result = self.normal.log_prob(pre_tanh) - log_det
        return result

    def rsample(self):
        pretanh = self.normal_mean + self.normal_std * self.standard_normal.sample()
        return torch.tanh(pretanh), pretanh

    
class ERE(object):
    def __init__(self, obs_dim, act_dim, time_horizon, capacity=int(1e6), decay_rate=0.996, min_count=5000):
        self.time_horizon = time_horizon
        self.capacity = capacity
        self.position = 0
        self.current_size = 0
        self.has_rolled_over = False
        self.initial_decay = decay_rate
        self.min_count = min_count
        self.c_values = []
        self.indices = []

        self.rewards = np.empty((capacity, 1))
        self.states = np.empty((capacity, obs_dim))
        self.actions = np.empty((capacity, act_dim))
        self.continuations = np.empty((capacity, 1))
        self.next_states = np.empty((capacity, obs_dim))

    def add(self, state, action, next_state, reward, done):
        self.states[self.position] = state
        self.actions[self.position] = action
        self.next_states[self.position] = next_state
        self.rewards[self.position] = reward
        self.continuations[self.position] = 1. - done

        self.position = (self.position + 1) % self.capacity

        if self.capacity > self.current_size + 1:
            self.current_size += 1
        else:
            self.current_size = self.capacity
            self.has_rolled_over = True 

    def fetch_sample(self, batch_size, timestep):
        # Decay rate calculation
        decay = self.calculate_decay(timestep)

        indices = np.array([self.compute_index(decay, k, batch_size) for k in range(batch_size)])

        actions = torch.tensor(self.actions[indices], dtype=torch.float, device=device)
        rewards = torch.tensor(self.rewards[indices], dtype=torch.float, device=device)
        states = torch.tensor(self.states[indices], dtype=torch.float, device=device)
        next_states = torch.tensor(self.next_states[indices], dtype=torch.float, device=device)
        not_done = torch.tensor(self.continuations[indices], dtype=torch.float, device=device)

        return states, actions, next_states, rewards, not_done

    def compute_index(self, decay, k, batch_size):
        count_adjusted = self.current_size * decay ** (k * 1000 / batch_size)
        effective_count = count_adjusted if count_adjusted > self.min_count else self.current_size

        if not self.has_rolled_over:
            return np.random.randint(self.current_size - effective_count, self.current_size)
        
        return np.random.randint(self.position + self.current_size - effective_count, self.position + self.current_size) % self.current_size

    def calculate_decay(self, timestep):
        return self.initial_decay + (1 - self.initial_decay) * timestep / self.time_horizon


class GradientStep(object):
    def __init__(
        self,
        *,
        actor,
        critic,
        critic_target,
        discount,
        tau,
        top_quantiles_to_drop,
        target_entropy,
        quantiles_total
    ):
        self.actor = actor
        self.critic = critic
        self.critic_target = critic_target
        self.log_alpha = torch.zeros((1,), requires_grad=True, device=device)
        self.quantiles_total = quantiles_total
        self.actor_optimizer = Adam(self.actor.parameters(), lr=3e-4)
        
        self.alpha_optimizer = Adam([self.log_alpha], lr=3e-4)
        self.critic_optimizer = Adam(self.critic.parameters(), lr=3e-4)
        self.discount, self.tau, self.top_quantiles_to_drop, self.target_entropy  = discount, tau, top_quantiles_to_drop,target_entropy


    def take_gradient_step(self, replay_buffer, t, batch_size=256):
        # Sample replay buffer
        state, action, next_state, reward, not_done = replay_buffer.fetch_sample(batch_size, t)
        alpha = torch.exp(self.log_alpha) #entropy temperature coefficient

        with torch.no_grad():
            # Action by the current actor for the sampled state
            new_next_action, next_log_pi = self.actor(next_state)

            # Compute and cut quantiles at the next state
            next_z = self.critic_target(next_state, new_next_action)  
            
            # Sort and drop top k quantiles to control overestimation.
            sorted_z, _ = torch.sort(next_z.reshape(batch_size, -1))
            sorted_z_part = sorted_z[:, :self.quantiles_total-self.top_quantiles_to_drop]

            # td error + entropy term
            target = reward + not_done * self.discount * (sorted_z_part - alpha * next_log_pi)
        
        # Get current Quantile estimates using action from the replay buffer
        cur_z = self.critic(state, action)
        critic_loss = quantile_huber_loss(cur_z, target)


        new_action, log_pi = self.actor(state)
        # detach the variable from the graph so we don't change it with other losses
        alpha_loss = -self.log_alpha * (log_pi + self.target_entropy).detach().mean()

        # Optimise critic
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # Update target networks
        for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        
        # Compute actor loss
        actor_loss = (alpha * log_pi - self.critic(state, new_action).mean(2).mean(1, keepdim=True)).mean()
        
        # Optimise the actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # Optimise the entropy coefficient
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()


def quantile_huber_loss(quantiles, samples, sum_over_quantiles = False):
    delta = samples[:, np.newaxis, np.newaxis, :] - quantiles[:, :, :, np.newaxis]  
    abs_delta = torch.abs(delta)
    huber_loss = torch.where(abs_delta > 1, abs_delta - 0.5, delta ** 2 * 0.5)
    n_quantiles = quantiles.shape[2]
    cumulative_prob = (torch.arange(n_quantiles, device=quantiles.device, dtype=torch.float) + 0.5) / n_quantiles
    cumulative_prob_shaped = cumulative_prob.view(1, 1, -1, 1)
    loss = (torch.abs(cumulative_prob_shaped - (delta < 0).float()) * huber_loss)

    # Summing over the quantile dimension 
    if sum_over_quantiles:
        loss = loss.sum(dim=-2).mean()
    else:
        loss = loss.mean()

    return loss


**Prepare the environment and wrap it to capture videos**

In [ ]:
%%capture
env = gym.make('BipedalWalker-v3')
#env = gym.make("BipedalWalkerHardcore-v3") # only attempt this when your agent has solved BipedalWalker-v3
env = gym.wrappers.Monitor(env, "./video", video_callable=lambda ep_id: ep_id%video_every == 0, force=True)

obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]

In [ ]:
print('The environment has {} observations and the agent can take {} actions'.format(obs_dim, act_dim))
print('The device is: {}'.format(device))

if device.type != 'cpu': print('It\'s recommended to train on the cpu for this')

In [ ]:
# hyperparameters
seed = 42
n_quantiles = 25
batch_size = 256
top_quantiles_to_drop_per_net = 2
tau = 0.005
n_nets = 5
discount = 0.98

env.seed(seed)
env.action_space.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)

# logging variables
ep_reward = 0
reward_list = []
plot_data = []
log_f = open("agent-log.txt","w+")

max_episodes = 2501
max_timesteps = 2000

# initialise agent
actor = Actor(obs_dim, act_dim).to(device)
critic = Critic(obs_dim, act_dim, n_quantiles, n_nets).to(device)
critic_target = copy.deepcopy(critic)
replay_buffer = ERE(obs_dim, act_dim, max_timesteps)
top_quantiles_to_drop = top_quantiles_to_drop_per_net * n_nets
gradient_step = GradientStep(actor=actor, critic=critic, critic_target=critic_target, top_quantiles_to_drop=top_quantiles_to_drop, discount=discount, tau=tau, target_entropy=-np.prod(env.action_space.shape).item(), quantiles_total=n_quantiles * n_nets)

actor.train()
state = env.reset()

# Main training loop
total_num_steps = 0
reward_list = []

for episode in range(1, max_episodes + 1):
    state = env.reset()
    ep_reward = 0  # Reset the episode reward at the start of each episode

    for t in range(max_timesteps):
        # Select the agent's action based on the current state
        action = actor.select_action(state)

        # Execute the action in the environment and observe the next state and reward
        next_state, reward, done, _ = env.step(action)
        replay_buffer.add(state, action, next_state, reward, done)
        state = next_state
        ep_reward += reward

        # Perform a gradient step if the replay buffer is sufficiently large
        if replay_buffer.current_size >= batch_size:
            gradient_step.take_gradient_step(replay_buffer, t, batch_size)

        # stop iterating when the episode finished
        if done or t==(max_timesteps-1):
            break

    # append the episode reward to the reward list
    reward_list.append(ep_reward)

    # do NOT change this logging code - it is used for automated marking!
    log_f.write('episode: {}, reward: {}\n'.format(episode, ep_reward))
    log_f.flush()
    ep_reward = 0

    # print reward data every so often - add a graph like this in your report
    if episode % plot_interval == 0:
        plot_data.append([episode, np.array(reward_list).mean(), np.array(reward_list).std()])
        reward_list = []
        # plt.rcParams['figure.dpi'] = 100
        plt.plot([x[0] for x in plot_data], [x[1] for x in plot_data], '-', color='tab:grey')
        plt.fill_between([x[0] for x in plot_data], [x[1]-x[2] for x in plot_data], [x[1]+x[2] for x in plot_data], alpha=0.2, color='tab:grey')
        plt.xlabel('Episode number')
        plt.ylabel('Episode reward')
        plt.show()
        disp.clear_output(wait=True)



Initial TD3 algorithm attempted (only for reference, not for submission): 

In [ ]:
# class Actor(nn.Module):
#     def __init__(self, obs_dim, act_dim):
#         super(Actor, self).__init__()
#         self.fc1 = nn.Linear(obs_dim, 400)
#         self.fc2 = nn.Linear(400, 300)
#         self.fc3 = nn.Linear(300, act_dim)

#     def forward(self, x):
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         return torch.tanh(self.fc3(x))

# class Critic(nn.Module):
#     def __init__(self, obs_dim, act_dim):
#         super(Critic, self).__init__()
#         self.fc1 = nn.Linear(obs_dim + act_dim, 400)
#         self.fc2 = nn.Linear(400, 300)
#         self.fc3 = nn.Linear(300, 1)

#     def forward(self, s, a):
#         x = torch.cat([s, a], dim=1)
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         return self.fc3(x)

# class TD3Agent:
#     def __init__(self, obs_dim, act_dim):
#         self.actor = Actor(obs_dim, act_dim).to(device)
#         self.actor_target = Actor(obs_dim, act_dim).to(device)
#         self.actor_target.load_state_dict(self.actor.state_dict())

#         self.critic1 = Critic(obs_dim, act_dim).to(device)
#         self.critic2 = Critic(obs_dim, act_dim).to(device)

#         self.critic1_target = Critic(obs_dim, act_dim).to(device)
#         self.critic1_target.load_state_dict(self.critic1.state_dict())
#         self.critic2_target = Critic(obs_dim, act_dim).to(device)
#         self.critic2_target.load_state_dict(self.critic2.state_dict())

#         self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=5e-4)
#         self.critic_optimizer = optim.Adam(list(self.critic1.parameters()) + list(self.critic2.parameters()), lr=1e-3)

#         self.replay_buffer = []
#         self.batch_size = 100
#         self.gamma = 0.99
#         self.tau = 0.005
#         self.policy_noise = 0.2
#         self.noise_clip = 0.5
#         self.policy_freq = 2

#     def sample_action(self, state):
#         state = torch.FloatTensor(state).to(device)
#         action = self.actor(state).cpu().data.numpy()
#         action += np.random.normal(0, 0.1, size=act_dim)
#         return np.clip(action, -1, 1)

#     def update(self, state, action, next_state, reward, done):
#         self.replay_buffer.append((state, action, next_state, reward, done))
#         if len(self.replay_buffer) < self.batch_size:
#             return

#         samples = random.sample(self.replay_buffer, self.batch_size)
#         state, action, next_state, reward, done = map(np.stack, zip(*samples))
#         state = torch.FloatTensor(state).to(device)
#         action = torch.FloatTensor(action).to(device)
#         next_state = torch.FloatTensor(next_state).to(device)
#         reward = torch.FloatTensor(reward).unsqueeze(1).to(device)
#         done = torch.FloatTensor(done).unsqueeze(1).to(device)

#         with torch.no_grad():
#             noise = (torch.randn_like(action) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
#             next_action = (self.actor_target(next_state) + noise).clamp(-1, 1)

#             target_Q1 = self.critic1_target(next_state, next_action)
#             target_Q2 = self.critic2_target(next_state, next_action)
#             target_Q = torch.min(target_Q1, target_Q2)
#             target_Q = reward + (1 - done) * self.gamma * target_Q

#         current_Q1 = self.critic1(state, action)
#         current_Q2 = self.critic2(state, action)
#         critic_loss = F.mse_loss(current_Q1, target_Q) + F.mse_loss(current_Q2, target_Q)
#         self.critic_optimizer.zero_grad()
#         critic_loss.backward()
#         self.critic_optimizer.step()

#         if len(self.replay_buffer) % self.policy_freq == 0:
#             actor_loss = -self.critic1(state, self.actor(state)).mean()
#             self.actor_optimizer.zero_grad()
#             actor_loss.backward()
#             self.actor_optimizer.step()

#             for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
#                 target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

#             for param, target_param in zip(self.critic1.parameters(), self.critic1_target.parameters()):
#                 target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

#             for param, target_param in zip(self.critic2.parameters(), self.critic2_target.parameters()):
#                 target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)